# 멤버 1번 소비 지표 및 행동 탐지 분석 (최종 수정본)

이 노트북은 과거(1~3월) 데이터와 오늘(4/1) 데이터를 비교하여 안정적 지표와 특이 행동을 심층 분석합니다.

In [47]:
import pandas as pd
import numpy as np
from datetime import datetime

# 1. 데이터 로드 및 전처리
df_past = pd.read_csv('../../../data/raw/csv/consumption_v1.csv')
df_today_full = pd.read_csv('./data_input_month.csv')

# 과거 데이터에서 멤버 1번만 추출
df_m1 = df_past[df_past['멤버 id'] == 1].copy()
df_m1['사용 시간'] = pd.to_datetime(df_m1['사용 시간'])
df_m1['date'] = df_m1['사용 시간'].dt.date

# 오늘 데이터 (2024-04-01) 추출
df_today_full['사용 시간'] = pd.to_datetime(df_today_full['사용 시간'])
df_today = df_today_full[df_today_full['사용 시간'].dt.strftime('%Y-%m-%d') == '2024-04-01'].copy()

# IQR 기반 클리핑 상하한선 설정
Q1 = df_m1['사용 금액'].quantile(0.25)
Q3 = df_m1['사용 금액'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = max(0, Q1 - 1.5 * IQR)
df_m1['사용 금액_clipped'] = df_m1['사용 금액'].clip(lower=lower_bound, upper=upper_bound)

print("✅ 데이터 로드 및 클리핑 설정 완료.")

✅ 데이터 로드 및 클리핑 설정 완료.


## [1] 클리핑 데이터 → 안정적 지표 분석
평소의 안정적인 소비 패턴(이상치 제외)과 오늘을 비교합니다.

In [48]:
# 과거 일평균(안정적) 계산
past_daily_stable_avg = df_m1.groupby('date')['사용 금액_clipped'].sum().mean()
today_total = df_today['사용 금액'].sum()

# 1. 평균 대비 및 증가율 계산
increase_rate = ((today_total - past_daily_stable_avg) / past_daily_stable_avg) * 100

print(f"[지표 1] 과거 일평균(안정): {past_daily_stable_avg:,.0f}원")
print(f"[지표 2] 오늘 총 지출액: {today_total:,.0f}원")
print(f"[지표 3] 평소 대비 지출 증가율: {increase_rate:.2f}%")

# 2. 카테고리 비율 비교
past_cat_ratio = df_m1.groupby('업종 카테고리')['사용 금액_clipped'].sum() / df_m1['사용 금액_clipped'].sum() * 100
today_cat_ratio = df_today.groupby('업종 카테고리')['사용 금액'].sum() / today_total * 100

cat_comparison = pd.DataFrame({
    '평소 비중(%)': past_cat_ratio,
    '오늘 비중(%)': today_cat_ratio
}).fillna(0)
cat_comparison['차이(pt)'] = cat_comparison['오늘 비중(%)'] - cat_comparison['평소 비중(%)']

print("\n--- 카테고리 비율 변화 ---")
display(cat_comparison.sort_values(by='차이(pt)', ascending=False))

[지표 1] 과거 일평균(안정): 51,014원
[지표 2] 오늘 총 지출액: 133,044원
[지표 3] 평소 대비 지출 증가율: 160.80%

--- 카테고리 비율 변화 ---


,평소 비중(%),오늘 비중(%),차이(pt)
업종 카테고리,,,
생활,1.470474,72.833048,71.362574
교통,4.741766,9.688524,4.946758
의료,8.093374,10.092902,1.999528
쇼핑,17.266950,0.000000,-17.266950
식비,68.427436,7.385527,-61.041910


## [2] 원본 데이터 → 행동 및 이상 탐지
원본 데이터를 기준으로 평소와 다른 특이 소비 행동을 탐지합니다.

In [49]:
# 1. 고액 소비 건 탐지 (과거 IQR 상한선 기준)
high_spending_items = df_today[df_today['사용 금액'] > upper_bound].copy()

# 2. 급증 소비 판단 (과거 원본 일평균 총액 대비)
past_daily_orig_avg = df_m1.groupby('date')['사용 금액'].sum().mean()
spike_ratio = today_total / past_daily_orig_avg

print(f"[탐지 1] 과거 원본 일평균(전체): {past_daily_orig_avg:,.0f}원")
if spike_ratio > 1.5:
    print(f"🚨 급증 소비 경보: 평소보다 {spike_ratio:.1f}배 높습니다!")
else:
    print("✅ 소비 규모가 평소 범위를 유지하고 있습니다.")

# 3. 특이 이벤트 내역 출력
print("\n--- [탐지 2] 오늘 발생한 특이 지출 내역 ---")
if not high_spending_items.empty:
    display(high_spending_items[['사용 시간', '결제 내역', '사용 금액', '업종 카테고리']])
else:
    print("오늘 발견된 특이 개별 지출 건이 없습니다.")

[탐지 1] 과거 원본 일평균(전체): 104,424원
✅ 소비 규모가 평소 범위를 유지하고 있습니다.

--- [탐지 2] 오늘 발생한 특이 지출 내역 ---


,사용 시간,결제 내역,사용 금액,업종 카테고리
2,2024-04-01 10:00:00,SKT통신비,65000,생활


## [3] 전날(3/31) 대비 소비 비교 분석
어제와 오늘의 지출 변화를 직접적으로 비교합니다.

In [50]:
# 전날(3/31) 데이터 추출
df_yesterday = df_m1[df_m1['date'].astype(str) == '2024-03-31'].copy()

yesterday_total = df_yesterday['사용 금액'].sum()
yesterday_count = len(df_yesterday)
today_count = len(df_today)

# 금액 및 건수 차이 계산
day_diff = today_total - yesterday_total
day_diff_rate = (day_diff / yesterday_total * 100) if yesterday_total > 0 else 0

print(f"--- 3/31(어제) vs 4/1(오늘) 비교 ---")
print(f"[지출액] 어제: {yesterday_total:,.0f}원 | 오늘: {today_total:,.0f}원 ({day_diff:+,.0f}원, {day_diff_rate:+.1f}%)")
print(f"[건수] 어제: {yesterday_count}건 | 오늘: {today_count}건 ({today_count - yesterday_count:+}건)")

# 주요 소비 카테고리 비교
if not df_yesterday.empty:
    yesterday_main_cat = df_yesterday.groupby('업종 카테고리')['사용 금액'].sum().idxmax()
    today_main_cat = df_today.groupby('업종 카테고리')['사용 금액'].sum().idxmax()
    print(f"[주 소비] 어제는 '{yesterday_main_cat}', 오늘은 '{today_main_cat}'에 가장 많이 썼습니다.")

--- 3/31(어제) vs 4/1(오늘) 비교 ---
[지출액] 어제: 48,531원 | 오늘: 133,044원 (+84,513원, +174.1%)
[건수] 어제: 6건 | 오늘: 9건 (+3건)
[주 소비] 어제는 '식비', 오늘은 '생활'에 가장 많이 썼습니다.


## [4] 시간별 소비 추세 분석 (Time-Slot Analysis)
하루를 5개 구간으로 나누어 언제 소비가 집중되는지 분석합니다.

In [51]:
# 시간대 분류 함수 정의
def get_time_slot(hour):
    if 0 <= hour < 6: return '1.새벽(00-06)'
    elif 6 <= hour < 11: return '2.오전(06-11)'
    elif 11 <= hour < 17: return '3.점심/오후(11-17)'
    elif 17 <= hour < 21: return '4.저녁(17-21)'
    else: return '5.밤/야식(21-24)'

# 오늘 데이터에 시간대 정보 추가
df_today['hour'] = df_today['사용 시간'].dt.hour
df_today['시간대'] = df_today['hour'].apply(get_time_slot)

# 과거 데이터(평균적 시간대 분포) 계산
df_m1['hour'] = df_m1['사용 시간'].dt.hour
df_m1['시간대'] = df_m1['hour'].apply(get_time_slot)
num_days = len(df_m1['date'].unique())
past_time_dist = df_m1.groupby('시간대')['사용 금액'].sum() / num_days

# 오늘 시간대별 합계 계산
today_time_dist = df_today.groupby('시간대')['사용 금액'].sum()

# 분석 결과 통합
time_analysis = pd.DataFrame({
    '오늘 지출액': today_time_dist,
    '평소 평균 지출액': past_time_dist
}).fillna(0)

print("--- 시간대별 소비 추세 분석 ---")
display(time_analysis.astype(int))

if not today_time_dist.empty:
    peak_slot = today_time_dist.idxmax()
    print(f"\n💡 오늘의 소비 피크 타임은 '{peak_slot}' 구간입니다.")

--- 시간대별 소비 추세 분석 ---


,오늘 지출액,평소 평균 지출액
시간대,,
2.오전(06-11),112895,9495
3.점심/오후(11-17),5471,68098
4.저녁(17-21),13428,16601
5.밤/야식(21-24),1250,10227



💡 오늘의 소비 피크 타임은 '2.오전(06-11)' 구간입니다.


- 과거 일평균과 오늘 지출액을 비교 (증가율 계산) 
- 카테고리별 비율 변화 (평소 비중, 오늘 비중) 
- 오늘 발생한 특이 지출 내역 
- 전날 대비 지출액 /  오늘 건수 / 주 소비 카테고리 비교 
- 시간별 소비 추세 분석 
